# 04 – Historical Feature Engineering

**Phase 3: Feature Engineering** · Group AI-39

This notebook transforms the football match data into one row per match with pre-match team-form features. It uses the project's shared functions in `src/db.py`, `src/preprocessing.py`, and `src/feature_engineering.py`; it does not use Team_Attributes.

## 1. Imports and configuration

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.db import load_matches
from src.preprocessing import (
    add_match_outcome,
    remove_excluded_columns,
    remove_warmup_season,
    sort_matches_chronologically,
)
from src.feature_engineering import (
    add_historical_features,
    build_team_long,
    calculate_training_imputation_values,
    impute_historical_features,
    to_match_level,
)

TEST_SEASON = "2015/2016"
EXPECTED_TRAIN_SEASONS = {
    "2009/2010", "2010/2011", "2011/2012",
    "2012/2013", "2013/2014", "2014/2015",
}
FEATURE_COLUMNS = [
    "Home_Form_Last_5", "Away_Form_Last_5",
    "Home_Avg_Goals_Last_5", "Away_Avg_Goals_Last_5",
    "Home_Avg_Conceded_Last_5", "Away_Avg_Conceded_Last_5",
    "Home_Win_Rate", "Away_Win_Rate",
]
FINAL_COLUMNS = [
    "id", "date", "season", "league_id",
    "home_team_api_id", "away_team_api_id", "match_outcome",
    *FEATURE_COLUMNS, "Form_Diff", "Abs_Form_Diff",
]
pd.set_option("display.max_columns", 30)

## 2. Load and prepare match rows

Load matches through `src.db.load_matches()`, then apply the shared chronological sort, target creation, and excluded-column removal. The first three transformations preserve all seasons so the warm-up season contributes to subsequent historical calculations.

In [3]:
matches = load_matches()
matches = sort_matches_chronologically(matches)
matches = add_match_outcome(matches)

columns_before_exclusion = set(matches.columns)
matches = remove_excluded_columns(matches)
excluded_columns = sorted(columns_before_exclusion - set(matches.columns))

print(f"All-season match table: {matches.shape[0]:,} rows × {matches.shape[1]} columns")
print(f"Excluded columns: {len(excluded_columns)}")
display(pd.Series(matches["match_outcome"].value_counts(), name="matches").to_frame())
assert matches["date"].is_monotonic_increasing
assert matches["id"].is_unique
assert matches["match_outcome"].isin(["Home Win", "Draw", "Away Win"]).all()

All-season match table: 25,979 rows × 16 columns
Excluded columns: 104


,matches
match_outcome,
Home Win,11917
Away Win,7466
Draw,6596


## 3. Build strictly historical team features

A prediction for a match can only use information available before kickoff. Including the current fixture's result in a form or win-rate statistic would leak the target into the predictors. The existing `add_historical_features()` function applies `shift(1)` before rolling or expanding calculations, so the current match is excluded. `build_team_long()` and `to_match_level()` handle the team/match reshaping.

In [4]:
team_matches = build_team_long(matches)
print(f"Team-match table: {team_matches.shape[0]:,} rows × {team_matches.shape[1]} columns")
display(team_matches.head())

Team-match table: 51,958 rows × 10 columns


,id,date,season,league_id,team_api_id,goals_for,goals_against,venue,points,win
0,15723,2008-08-10,2008/2009,15722,1601,0,0,away,1,0
1,15810,2008-08-16,2008/2009,15722,1601,2,1,home,3,1
2,15898,2008-08-22,2008/2009,15722,1601,2,1,home,3,1
3,15916,2008-08-30,2008/2009,15722,1601,0,3,away,0,0
4,15922,2008-09-12,2008/2009,15722,1601,2,0,home,3,1


In [5]:
team_matches = add_historical_features(team_matches)
match_features = to_match_level(team_matches)

matches_with_features = matches.merge(match_features, on="id", how="inner", validate="one_to_one")
print(f"Match-level table before warm-up removal: {matches_with_features.shape}")
display(matches_with_features[[
    "id", "date", "season", "Home_Form_Last_5", "Away_Form_Last_5",
    "Home_Avg_Goals_Last_5", "Away_Avg_Goals_Last_5",
    "Home_Win_Rate", "Away_Win_Rate",
]].head())

Match-level table before warm-up removal: (25979, 26)


,id,date,season,Home_Form_Last_5,Away_Form_Last_5,Home_Avg_Goals_Last_5,Away_Avg_Goals_Last_5,Home_Win_Rate,Away_Win_Rate
0,24559,2008-07-18,2008/2009,NaN,NaN,NaN,NaN,NaN,NaN
1,24560,2008-07-19,2008/2009,NaN,NaN,NaN,NaN,NaN,NaN
2,24561,2008-07-20,2008/2009,NaN,NaN,NaN,NaN,NaN,NaN
3,24562,2008-07-20,2008/2009,NaN,NaN,NaN,NaN,NaN,NaN
4,24613,2008-07-23,2008/2009,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Remove the warm-up season

Historical calculations were performed using all seasons. Now remove 2008/2009, allowing its match history to contribute to the next season while excluding warm-up fixtures from the modelling dataset.

In [6]:
features = remove_warmup_season(matches_with_features)
print(f"Rows after removing 2008/2009: {len(features):,}")
display(features["season"].value_counts().sort_index().rename("matches").to_frame())
assert "2008/2009" not in set(features["season"])
assert set(features["season"]) == EXPECTED_TRAIN_SEASONS | {TEST_SEASON}

Rows after removing 2008/2009: 22,653


,matches
season,
2009/2010,3230
2010/2011,3260
2011/2012,3220
2012/2013,3260
2013/2014,3032
2014/2015,3325
2015/2016,3326


## 5. Training-only missing-value imputation

Use 2009/2010–2014/2015 as the training period and 2015/2016 as the chronological test season. Calculate medians with the shared `calculate_training_imputation_values()` function; its `test_season` argument excludes 2015/2016 from the calculation. Apply those same training-derived values to missing historical features in both periods. This step imputes historical predictors only; it does not split or reorder the data.

In [7]:
training_seasons = set(features.loc[features["season"] != TEST_SEASON, "season"])
test_seasons = set(features.loc[features["season"] == TEST_SEASON, "season"])
assert training_seasons == EXPECTED_TRAIN_SEASONS
assert test_seasons == {TEST_SEASON}

training_fill_values = calculate_training_imputation_values(
    features,
    test_season=TEST_SEASON,
)
print("Training-derived median values (test season excluded):")
display(pd.Series(training_fill_values, name="training median").to_frame())

features = impute_historical_features(features, training_fill_values)
print(f"Missing feature values after imputation: {int(features[FEATURE_COLUMNS].isna().sum().sum())}")

Training-derived median values (test season excluded):


,training median
Home_Form_Last_5,7.000000
Away_Form_Last_5,7.000000
Home_Avg_Goals_Last_5,1.200000
Away_Avg_Goals_Last_5,1.200000
Home_Avg_Conceded_Last_5,1.400000
Away_Avg_Conceded_Last_5,1.200000
Home_Win_Rate,0.454545
Away_Win_Rate,0.260870


Missing feature values after imputation: 0


## 6. Leakage-prevention spot check

For five fixtures with complete pre-imputation form, independently list each team's five most recent team-match records strictly preceding the fixture in `(date, id)` order. Compare their points sum with the stored form. The current fixture's goals are shown for context, but its own team-match row is excluded from the prior-record selection.

In [12]:
# ============================================================
# MANUAL SPOT-CHECK: HISTORICAL FORM AND LEAKAGE PREVENTION
# ============================================================

# Exclude the warm-up season for this validation.
pre_imputation = matches_with_features.loc[
    matches_with_features["season"] != "2008/2009"
].copy()

# Select five matches with complete historical form values.
spot_matches = (
    pre_imputation.dropna(
        subset=["Home_Form_Last_5", "Away_Form_Last_5"]
    )
    .sort_values(["date", "id"])
    .head(5)
)

spot_rows = []
previous_match_rows = []

# Check both teams in each selected match.
for match in spot_matches.itertuples(index=False):

    for side, team_id, stored_form in (
        ("Home", match.home_team_api_id, match.Home_Form_Last_5),
        ("Away", match.away_team_api_id, match.Away_Form_Last_5),
    ):

        # Select only matches played before the current match.
        prior = team_matches.loc[
            (team_matches["team_api_id"] == team_id)
            & (
                (team_matches["date"] < match.date)
                | (
                    (team_matches["date"] == match.date)
                    & (team_matches["id"] < match.id)
                )
            )
        ].sort_values(["date", "id"]).tail(5)

        # Recalculate historical form from the previous five matches.
        expected_form = prior["points"].sum()

        # Check that exactly five earlier matches were used.
        assert len(prior) == 5, (
            f"Expected 5 previous matches: "
            f"match={match.id}, team={team_id}"
        )

        # Compare the independently recalculated form with the stored value.
        assert np.isclose(stored_form, expected_form), (
            f"Form mismatch: match={match.id}, team={team_id}, "
            f"stored={stored_form}, expected={expected_form}"
        )

        # Ensure the current match is not part of its own history.
        current_match_excluded = match.id not in set(prior["id"])

        assert current_match_excluded, (
            f"Current match {match.id} appears in its own history."
        )

        # Store the summary result for this team in this match.
        spot_rows.append({
            "current_match_id": match.id,
            "date": match.date,
            "team_side": side,
            "team_api_id": team_id,
            "current_match_goals": (
                f"{match.home_team_goal}-{match.away_team_goal}"
            ),
            "earlier_matches_used": len(prior),
            "recalculated_form": expected_form,
            "stored_form": stored_form,
            "match_row_excluded": current_match_excluded,
        })

        # Store each previous match separately for manual inspection.
        for previous_number, row in enumerate(
            prior.itertuples(index=False), start=1
        ):
            previous_match_rows.append({
                "current_match_id": match.id,
                "team_side": side,
                "team_api_id": team_id,
                "previous_match_number": previous_number,
                "previous_match_id": row.id,
                "previous_match_date": row.date,
                "goals_for": row.goals_for,
                "goals_against": row.goals_against,
                "points": row.points,
            })

# Create the validation tables.
spot_check = pd.DataFrame(spot_rows)
previous_matches_audit = pd.DataFrame(previous_match_rows)

# Keep column contents from being shortened with an ellipsis.
pd.set_option("display.max_colwidth", None)

print("TABLE 1: SPOT-CHECK SUMMARY")
display(spot_check)

print("TABLE 2: EACH OF THE FIVE PREVIOUS MATCHES")
display(previous_matches_audit)

# ============================================================
# FINAL VALIDATION ASSERTIONS
# ============================================================

assert len(spot_matches) == 5
assert len(spot_check) == 10
assert len(previous_matches_audit) == 50

assert spot_check["earlier_matches_used"].eq(5).all()
assert spot_check["match_row_excluded"].all()

assert np.isclose(
    spot_check["recalculated_form"],
    spot_check["stored_form"],
).all()

print("PASS: Five matches checked for both home and away teams.")
print("PASS: Exactly five earlier matches were used for each team.")
print("PASS: Previous-match details are available for manual inspection.")
print("PASS: Recalculated form matches the stored feature.")
print("PASS: Current matches are excluded from their own history.")

TABLE 1: SPOT-CHECK SUMMARY


,current_match_id,date,team_side,team_api_id,current_match_goals,earlier_matches_used,recalculated_form,stored_form,match_row_excluded
0,24738,2009-07-11,Home,9930,1-0,5,7,7.0,True
1,24738,2009-07-11,Away,9956,1-0,5,8,8.0,True
2,24739,2009-07-12,Home,6493,1-1,5,7,7.0,True
3,24739,2009-07-12,Away,7955,1-1,5,9,9.0,True
4,24741,2009-07-14,Home,10243,2-3,5,10,10.0,True
5,24741,2009-07-14,Away,10192,2-3,5,12,12.0,True
6,24793,2009-07-18,Home,7955,3-0,5,7,7.0,True
7,24793,2009-07-18,Away,10243,3-0,5,9,9.0,True
8,24795,2009-07-19,Home,9931,2-1,5,6,6.0,True
9,24795,2009-07-19,Away,9930,2-1,5,7,7.0,True


TABLE 2: EACH OF THE FIVE PREVIOUS MATCHES


,current_match_id,team_side,team_api_id,previous_match_number,previous_match_id,previous_match_date,goals_for,goals_against,points
0,24738,Home,9930,1,24683,2009-05-09,3,1,3
1,24738,Home,9930,2,24690,2009-05-13,1,1,1
2,24738,Home,9930,3,24695,2009-05-16,0,1,0
3,24738,Home,9930,4,24701,2009-05-24,1,3,0
4,24738,Home,9930,5,24703,2009-05-30,2,0,3
5,24738,Away,9956,1,24687,2009-05-10,4,1,3
6,24738,Away,9956,2,24690,2009-05-13,1,1,1
7,24738,Away,9956,3,24696,2009-05-17,1,1,1
8,24738,Away,9956,4,24699,2009-05-24,4,1,3
9,24738,Away,9956,5,24707,2009-05-29,1,2,0


PASS: Five matches checked for both home and away teams.
PASS: Exactly five earlier matches were used for each team.
PASS: Previous-match details are available for manual inspection.
PASS: Recalculated form matches the stored feature.
PASS: Current matches are excluded from their own history.


## 7. Final dataset and validation

Keep the requested 17 match-level columns. Current-match goals are retained only in the temporary preprocessing inputs needed to construct outcomes and historical team records; they are not included in the final predictors.

In [9]:
final_df = features.loc[:, FINAL_COLUMNS].copy()

assert len(final_df.columns) == 17
assert final_df["id"].is_unique
assert final_df["date"].is_monotonic_increasing
assert final_df.isna().sum().sum() == 0
assert not {"home_team_goal", "away_team_goal"}.intersection(final_df.columns)
assert np.allclose(
    final_df["Form_Diff"],
    final_df["Home_Form_Last_5"] - final_df["Away_Form_Last_5"],
)
assert np.allclose(final_df["Abs_Form_Diff"], final_df["Form_Diff"].abs())
assert final_df["season"].max() == TEST_SEASON

validation = pd.Series({
    "rows": len(final_df),
    "columns": len(final_df.columns),
    "missing values": int(final_df.isna().sum().sum()),
    "duplicate match IDs": int(final_df["id"].duplicated().sum()),
    "chronological order": final_df["date"].is_monotonic_increasing,
    "Form_Diff correct": np.allclose(
        final_df["Form_Diff"],
        final_df["Home_Form_Last_5"] - final_df["Away_Form_Last_5"],
    ),
    "Abs_Form_Diff correct": np.allclose(
        final_df["Abs_Form_Diff"], final_df["Form_Diff"].abs()
    ),
}, name="validation result")
display(validation.to_frame())
display(final_df.head())

,validation result
rows,22653
columns,17
missing values,0
duplicate match IDs,0
chronological order,True
Form_Diff correct,True
Abs_Form_Diff correct,True


,id,date,season,league_id,home_team_api_id,away_team_api_id,match_outcome,Home_Form_Last_5,Away_Form_Last_5,Home_Avg_Goals_Last_5,Away_Avg_Goals_Last_5,Home_Avg_Conceded_Last_5,Away_Avg_Conceded_Last_5,Home_Win_Rate,Away_Win_Rate,Form_Diff,Abs_Form_Diff
0,24738,2009-07-11,2009/2010,24558,9930,9956,Home Win,7.0,8.0,1.4,2.2,1.2,1.2,0.500000,0.222222,-1.0,1.0
1,24739,2009-07-12,2009/2010,24558,6493,7955,Draw,7.0,9.0,1.4,1.8,1.2,1.2,0.444444,0.166667,-2.0,2.0
2,24740,2009-07-12,2009/2010,24558,10190,9931,Home Win,7.0,6.0,1.2,1.4,1.4,2.2,0.454545,0.555556,1.0,1.0
3,24741,2009-07-14,2009/2010,24558,10243,10192,Away Win,10.0,12.0,2.0,2.4,1.6,1.0,0.722222,0.555556,-2.0,2.0
4,24793,2009-07-18,2009/2010,24558,7955,10243,Home Win,7.0,9.0,1.2,2.2,1.0,2.0,0.388889,0.611111,-2.0,2.0


## 8. Save the engineered dataset

Write the validated feature table to the project's processed-data location. Re-running this notebook from the repository root rebuilds and replaces this generated CSV.

In [10]:
output_path = ROOT / "data" / "processed" / "features.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(output_path, index=False)
print(f"Saved {len(final_df):,} rows and {len(final_df.columns)} columns to {output_path.relative_to(ROOT)}")

Saved 22,653 rows and 17 columns to data\processed\features.csv


## 9. Generated dataset summary

In [11]:
print(f"Dataset shape: {final_df.shape}")
print(f"Date range: {final_df['date'].min().date()} to {final_df['date'].max().date()}")
print(f"Output file: {output_path}")
display(final_df.head(10))
display(final_df["match_outcome"].value_counts().rename("matches").to_frame())

Dataset shape: (22653, 17)
Date range: 2009-07-11 to 2016-05-25
Output file: c:\Users\UMAIR\Desktop\football-ml-project\data\processed\features.csv


,id,date,season,league_id,home_team_api_id,away_team_api_id,match_outcome,Home_Form_Last_5,Away_Form_Last_5,Home_Avg_Goals_Last_5,Away_Avg_Goals_Last_5,Home_Avg_Conceded_Last_5,Away_Avg_Conceded_Last_5,Home_Win_Rate,Away_Win_Rate,Form_Diff,Abs_Form_Diff
0,24738,2009-07-11,2009/2010,24558,9930,9956,Home Win,7.0,8.0,1.4,2.2,1.2,1.2,0.500000,0.222222,-1.0,1.0
1,24739,2009-07-12,2009/2010,24558,6493,7955,Draw,7.0,9.0,1.4,1.8,1.2,1.2,0.444444,0.166667,-2.0,2.0
2,24740,2009-07-12,2009/2010,24558,10190,9931,Home Win,7.0,6.0,1.2,1.4,1.4,2.2,0.454545,0.555556,1.0,1.0
3,24741,2009-07-14,2009/2010,24558,10243,10192,Away Win,10.0,12.0,2.0,2.4,1.6,1.0,0.722222,0.555556,-2.0,2.0
4,24793,2009-07-18,2009/2010,24558,7955,10243,Home Win,7.0,9.0,1.2,2.2,1.0,2.0,0.388889,0.611111,-2.0,2.0
5,24794,2009-07-18,2009/2010,24558,10192,10190,Draw,15.0,7.0,2.8,1.2,1.0,1.2,0.666667,0.260870,8.0,8.0
6,24795,2009-07-19,2009/2010,24558,9931,9930,Home Win,6.0,7.0,1.2,1.0,2.0,1.0,0.666667,0.111111,-1.0,1.0
7,24796,2009-07-19,2009/2010,24558,6493,10199,Away Win,5.0,5.0,1.2,1.2,1.2,2.0,0.421053,0.055556,0.0,0.0
8,24797,2009-07-19,2009/2010,24558,9956,10179,Home Win,5.0,7.0,1.4,2.0,1.2,1.8,0.444444,0.166667,-2.0,2.0
9,24742,2009-07-22,2009/2010,24558,10179,10199,Home Win,7.0,7.0,2.0,1.4,1.6,2.0,0.333333,0.105263,0.0,0.0


,matches
match_outcome,
Home Win,10351
Away Win,6537
Draw,5765
